# Evaluate results

In [1]:
import re
import pandas as pd
from typing import Optional

def extract_ground_truth(df: pd.DataFrame, task: str) -> Optional[pd.Series]:
    if task == "t":
        return df["T14"].astype(int)
    elif task == "n":
        return df["N03"].astype(int)
    return None

_T_RE = re.compile(r"T\s*([1-4])(?:\s*[A-D])?(?!\s*\d)", re.IGNORECASE)
_N_RE = re.compile(r"N\s*([0-3])(?:\s*[A-D])?(?!\s*\d)", re.IGNORECASE)

def _first_pred_index_from_text(task: str, pred_text) -> Optional[int]:
    """
      - T: {T1..T4} -> {0..3}
      - N: {N0..N3} -> {0..3}
    """
    if pd.isna(pred_text):
        return None
    s = str(pred_text)

    if task.lower() == "t":
        m = _T_RE.search(s)
        return (int(m.group(1)) - 1) if m else None
    else:
        m = _N_RE.search(s)
        return int(m.group(1)) if m else None

def _label_name(task: str, k: int) -> str:
    return f"T{k+1}" if task.lower() == "t" else f"N{k}"

class _Float3(float):
    """float that renders with 3 decimal places when printed/repr'ed."""
    def __new__(cls, value):
        return super().__new__(cls, value)
    def __repr__(self):
        return f"{float(self):.3f}"
    __str__ = __repr__

def _format_floats_3dp(obj):
    """Recursively wrap floats so they print with 3 decimals."""
    if isinstance(obj, float):
        return _Float3(obj)
    if isinstance(obj, dict):
        return {k: _format_floats_3dp(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_format_floats_3dp(v) for v in obj]
    return obj


def compute_prf_firstmatch(task: str,
                           y_true_idx: pd.Series,
                           y_pred_texts: pd.Series):
    """
    Evaluation rule (single-label):
    - pred_idx = index of the first T/N token in the prediction string (None if not found)
    - Per-sample accounting:
        - pred_idx is None: FN[gt] += 1 (no prediction)
        - pred_idx == gt : TP[gt] += 1
        - pred_idx != gt : FP[pred_idx] += 1; FN[gt] += 1
    Returns:
    - per_class: tp/fp/fn, precision/recall/f1, support (= #ground-truth instances) for each label
    - macro/micro P/R/F1
    - accuracy, coverage (= fraction with a prediction)
    - overall_support (= total #samples)
    - totals: TP/FP/FN, n (= total #samples), n_pred (= #samples with a prediction)
    """
    K = 4
    tp = [0]*K
    fp = [0]*K
    fn = [0]*K
    support = [0]*K  # per-label ground-truth count (= distribution of y_true)

    n = int(len(y_true_idx))
    n_pred = 0
    correct = 0

    for gt, pred_text in zip(y_true_idx.astype(int), y_pred_texts):
        gt = int(gt)
        support[gt] += 1

        pred_idx = _first_pred_index_from_text(task, pred_text)
        if pred_idx is None:
            fn[gt] += 1
        else:
            n_pred += 1
            if int(pred_idx) == gt:
                tp[gt] += 1
                correct += 1
            else:
                fp[int(pred_idx)] += 1
                fn[gt] += 1

    # per-class metrics
    per_class = []
    precisions, recalls, f1s = [], [], []
    for k in range(K):
        p = tp[k] / (tp[k] + fp[k]) if (tp[k] + fp[k]) > 0 else 0.0
        r = tp[k] / (tp[k] + fn[k]) if (tp[k] + fn[k]) > 0 else 0.0
        f = (2*p*r/(p+r)) if (p+r) > 0 else 0.0
        per_class.append({
            "label": _label_name(task, k),
            "tp": tp[k], "fp": fp[k], "fn": fn[k],
            "support": support[k],        # per-label ground-truth count
            "precision": p, "recall": r, "f1": f,
        })
        precisions.append(p); recalls.append(r); f1s.append(f)

    macro_p = sum(precisions)/K
    macro_r = sum(recalls)/K
    macro_f1 = sum(f1s)/K

    micro_p = (correct / n_pred) if n_pred > 0 else 0.0
    micro_r = (correct / n) if n > 0 else 0.0
    micro_f1 = (2*micro_p*micro_r/(micro_p+micro_r)) if (micro_p+micro_r) > 0 else 0.0

    accuracy = correct / n if n > 0 else 0.0
    coverage = n_pred / n if n > 0 else 0.0

    result = {
        "per_class": per_class,
        "overall":{
        "macro": {"precision": macro_p, "recall": macro_r, "f1": macro_f1},
        "micro": {"precision": micro_p, "recall": micro_r, "f1": micro_f1},
        "accuracy": accuracy,
        "coverage": f"{_format_floats_3dp(coverage)} ({n_pred}/{n})",
        "overall_support": n,      # overall ground-truth count (= #samples)
        "totals": {
            "TP": int(sum(tp)),
            "FP": int(sum(fp)),
            "FN": int(sum(fn)),
            "n": n,
            "n_pred": n_pred
            }
        }   
    }
    return _format_floats_3dp(result)

def compute_prf_firstmatch_from_df(df: pd.DataFrame, task: str, method: str):
    """
    - task: 't' or 'n'
    - method: 'zscot' | 'rag' | 'kewrag' | 'kewltm'
    """
    y_true_idx = extract_ground_truth(df, task).astype(int)
    pred_col = f"{method}_stage"
    if pred_col not in df.columns:
        raise ValueError(f"Column '{pred_col}' not found.")
    return compute_prf_firstmatch(task, y_true_idx, df[pred_col])

df = pd.read_csv("/home/yl3427/cylab/selfCorrectionAgent/runs3/BLCA_T14N03__rag__t__mistralai_Mixtral-8x7B-Instruct-v0.1__seed42__20250909_085627.csv")
compute_prf_firstmatch_from_df(df, task="t", method="rag")['overall']


{'macro': {'precision': 0.737, 'recall': 0.780, 'f1': 0.746},
 'micro': {'precision': 0.896, 'recall': 0.896, 'f1': 0.896},
 'accuracy': 0.896,
 'coverage': '1.000 (345/345)',
 'overall_support': 345,
 'totals': {'TP': 309, 'FP': 36, 'FN': 36, 'n': 345, 'n_pred': 345}}

In [2]:
from pathlib import Path
import re
import pandas as pd

# capture method and task from file name, such as "__kewltm__t__"
PAT = re.compile(r"__(?P<method>[a-z0-9]+)__(?P<task>[tn])__")

def tcga_from_filename(path: Path):
    m = PAT.search(path.name) 
    if not m:
        raise ValueError(f"Unexpected filename: {path.name}")
    return m.group("method"), m.group("task")

In [9]:
per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs_med")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        results = compute_prf_firstmatch_from_df(df, task=task, method=method)
        print(f"type: {cancer_type}, method: {method}, task: {task}")
        print(f"micro f1 score: {results['overall']['micro']}, coverage: {results['overall']['coverage']}")
        print()

type: ACC, method: kewltm, task: t
micro f1 score: {'precision': 0.412, 'recall': 0.393, 'f1': 0.402}, coverage: 0.952 (80/84)

type: ACC, method: kewrag, task: t
micro f1 score: {'precision': 0.576, 'recall': 0.557, 'f1': 0.566}, coverage: 0.966 (85/88)

type: ACC, method: rag, task: t
micro f1 score: {'precision': 0.688, 'recall': 0.625, 'f1': 0.655}, coverage: 0.909 (80/88)

type: ACC, method: zscot, task: t
micro f1 score: {'precision': 0.598, 'recall': 0.591, 'f1': 0.594}, coverage: 0.989 (87/88)

type: BLCA, method: kewltm, task: t
micro f1 score: {'precision': 0.885, 'recall': 0.841, 'f1': 0.863}, coverage: 0.951 (312/328)

type: BLCA, method: kewrag, task: t
micro f1 score: {'precision': 0.928, 'recall': 0.823, 'f1': 0.873}, coverage: 0.887 (306/345)

type: BLCA, method: rag, task: t
micro f1 score: {'precision': 0.879, 'recall': 0.843, 'f1': 0.861}, coverage: 0.959 (331/345)

type: BLCA, method: zscot, task: t
micro f1 score: {'precision': 0.913, 'recall': 0.765, 'f1': 0.833},

In [11]:
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "n":
        results = compute_prf_firstmatch_from_df(df, task=task, method=method)
        print(f"type: {cancer_type}, method: {method}, task: {task}")
        print(f"micro f1 score: {results['overall']['micro']['f1']}, coverage: {results['overall']['coverage']}")
        print()

type: ACC, method: kewltm, task: n
micro f1 score: 0.456, coverage: 0.357 (30/84)

type: ACC, method: kewrag, task: n
micro f1 score: 0.695, coverage: 0.602 (53/88)

type: ACC, method: rag, task: n
micro f1 score: 0.783, coverage: 0.830 (73/88)

type: ACC, method: zscot, task: n
micro f1 score: 0.740, coverage: 0.750 (66/88)

type: BLCA, method: kewltm, task: n
micro f1 score: 0.894, coverage: 0.972 (312/321)

type: BLCA, method: kewrag, task: n
micro f1 score: 0.908, coverage: 0.962 (325/338)

type: BLCA, method: rag, task: n
micro f1 score: 0.892, coverage: 0.976 (330/338)

type: BLCA, method: zscot, task: n
micro f1 score: 0.885, coverage: 0.926 (313/338)



# Examine how much the coverage increased after repair

In [5]:
from pathlib import Path
import re
import pandas as pd

PAT = re.compile(r"__(?P<method>[a-z0-9]+)__(?P<task>[tn])__")

def tcga_from_filename(path: Path):
    m = PAT.search(path.name) 
    if not m:
        raise ValueError(f"Unexpected filename: {path.name}")
    return m.group("method"), m.group("task")

def stage_to_idx(task: str, stage) -> Optional[int]:
    if pd.isna(stage):
        return None
    s = str(stage)
    if not s.strip():
        return None

    if task.lower() == "t":
        m = _T_RE.search(s)
        return (int(m.group(1)) - 1) if m else None
    elif task.lower() == "n":
        m = _N_RE.search(s)
        return int(m.group(1)) if m else None
    return None

In [6]:
per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len(true_col), len([v for v in pred_col if v is not None]))


cancer_type: ACC, method: kewltm, task: n
84 26
cancer_type: ACC, method: kewltm, task: t
84 83
cancer_type: ACC, method: kewrag, task: n
88 49
cancer_type: ACC, method: kewrag, task: t
88 84
cancer_type: ACC, method: rag, task: n
88 56
cancer_type: ACC, method: rag, task: t
88 82
cancer_type: ACC, method: zscot, task: n
88 76
cancer_type: ACC, method: zscot, task: t
88 86
cancer_type: BLCA, method: kewltm, task: n
321 312
cancer_type: BLCA, method: kewltm, task: t
328 325
cancer_type: BLCA, method: kewrag, task: n
338 308
cancer_type: BLCA, method: kewrag, task: t
345 342
cancer_type: BLCA, method: rag, task: n
338 319
cancer_type: BLCA, method: rag, task: t
345 342
cancer_type: BLCA, method: zscot, task: n
338 323
cancer_type: BLCA, method: zscot, task: t
345 339
cancer_type: BRCA, method: kewltm, task: n
760 699
cancer_type: BRCA, method: kewltm, task: t
979 960
cancer_type: BRCA, method: kewrag, task: n
800 749
cancer_type: BRCA, method: kewrag, task: t
1031 1004
cancer_type: BRCA,

In [7]:
per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs2")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len(true_col), len([v for v in pred_col if v is not None]))


cancer_type: ACC, method: kewltm, task: n
84 28
cancer_type: ACC, method: kewltm, task: t
84 84
cancer_type: ACC, method: kewrag, task: n
88 61
cancer_type: ACC, method: kewrag, task: t
88 85
cancer_type: ACC, method: rag, task: n
88 67
cancer_type: ACC, method: rag, task: t
88 86
cancer_type: ACC, method: zscot, task: n
88 81
cancer_type: ACC, method: zscot, task: t
88 87
cancer_type: BLCA, method: kewltm, task: n
321 321
cancer_type: BLCA, method: kewltm, task: t
328 328
cancer_type: BLCA, method: kewrag, task: n
338 317
cancer_type: BLCA, method: kewrag, task: t
345 343
cancer_type: BLCA, method: rag, task: n
338 330
cancer_type: BLCA, method: rag, task: t
345 344
cancer_type: BLCA, method: zscot, task: n
338 330
cancer_type: BLCA, method: zscot, task: t
345 342
cancer_type: BRCA, method: kewltm, task: n
760 723
cancer_type: BRCA, method: kewltm, task: t
979 969
cancer_type: BRCA, method: kewrag, task: n
800 777
cancer_type: BRCA, method: kewrag, task: t
1031 1021
cancer_type: BRCA,

In [8]:
per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs3")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len([v for v in true_col if v is not None]), len([v for v in pred_col if v is not None]))

cancer_type: ACC, method: kewltm, task: n
84 32
cancer_type: ACC, method: kewltm, task: t
84 84
cancer_type: ACC, method: kewrag, task: n
88 70
cancer_type: ACC, method: kewrag, task: t
88 87
cancer_type: ACC, method: rag, task: n
88 79
cancer_type: ACC, method: rag, task: t
88 88
cancer_type: ACC, method: zscot, task: n
88 86
cancer_type: ACC, method: zscot, task: t
88 88
cancer_type: BLCA, method: kewltm, task: n
321 321
cancer_type: BLCA, method: kewltm, task: t
328 328
cancer_type: BLCA, method: kewrag, task: n
338 322
cancer_type: BLCA, method: kewrag, task: t
345 343
cancer_type: BLCA, method: rag, task: n
338 336
cancer_type: BLCA, method: rag, task: t
345 345
cancer_type: BLCA, method: zscot, task: n
338 337
cancer_type: BLCA, method: zscot, task: t
345 344
cancer_type: BRCA, method: kewltm, task: n
760 750
cancer_type: BRCA, method: kewltm, task: t
979 976
cancer_type: BRCA, method: kewrag, task: n
800 794
cancer_type: BRCA, method: kewrag, task: t
1031 1029
cancer_type: BRCA,